# 07 Gold-Schicht und Datenqualität

## Zweck
Analysefertige Gold-Datensätze erstellen, tabellenübergreifende Datenqualitätsprüfungen durchführen und eine saubere Übergabe an Phase 8 vorbereiten.

Historische EEA-Daten und aktuelle Open-Meteo-Kontextdaten bleiben methodisch getrennt. Spark- und Kafka-Infrastruktur werden sichtbar geprüft. Gemeinsamer FH-Clusterspeicher wird nur behauptet, wenn eine explizite Spark-Schreib- und Leseprobe erfolgreich ist.


## Eingaben

- `data/silver/city_reference.parquet`
- `data/silver/city_metadata.parquet`
- `data/silver/eea_city_daily.parquet`
- Bevorzugte Live-Eingabe: `data/silver/open_meteo_city_hourly/` aus Notebook `06`
- Lokaler funktionaler Fallback: `data/bronze/open_meteo_raw/open_meteo_air_quality_events.jsonl` aus Notebook `05`

## Ausgaben

- `data/gold/city_air_quality_daily_summary.parquet`
- `data/gold/pollutant_ranking_by_city.parquet`
- `data/gold/city_context_air_quality.parquet`
- `data/gold/live_air_quality_latest.parquet`
- `data/gold/data_quality_summary.parquet`


## Verwendete Technologien
Python, pandas, pyarrow, optionale PySpark-Speicherprobe, Socket-Erreichbarkeitsprüfungen und Parquet. Spark-Structured-Streaming bleibt in Notebook `06` umgesetzt.


## Konfiguration

Die Gold-Schicht schreibt für reproduzierbare Läufe lokale Parquet-Dateien. Das Notebook prüft Kafka-Broker und Spark-Master per TCP. `RUN_PHASE7_SPARK_STORAGE_PROBE=true` wird nur gesetzt, wenn Spark installiert ist und der konfigurierte Master `DATA_DIR` schreiben und lesen soll.

Falls lokal kein Silver-Streaming-Parquet aus Phase 6 vorhanden ist, rekonstruiert das Notebook den Live-Snapshot aus validierten JSONL-Ereignissen aus Phase 5. Dies ist ein expliziter funktionaler Fallback, kein Kafka-zu-Spark-Nachweis.


### Repository-Stammverzeichnis bestimmen

Das Notebook lädt `.env` aus dem Repository und unterstützt die Ausführung aus dem Stammverzeichnis sowie aus `notebooks/`.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime, timezone
import json
import os
import shutil
import socket
import pandas as pd

_cwd = Path.cwd().resolve()
_candidate_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
load_dotenv(_candidate_root / ".env")
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", _candidate_root)).resolve()

print({"project_root": str(PROJECT_ROOT)})


### Wiederverwendbare projektbezogene Pfade definieren

Der Helfer löst relative Datenpfade unterhalb des Repository-Stammverzeichnisses auf und erhält absolute Pfade zu gemeinsamem Speicher.


In [ ]:
def project_path(env_name: str, default: str) -> Path:
    path = Path(os.getenv(env_name, default))
    return path if path.is_absolute() else PROJECT_ROOT / path

print("OK: project_path definiert")


### Silver- und Gold-Datensätze deklarieren

Alle Ein- und Ausgabeverträge sind an einer Stelle sichtbar. Dadurch bleibt die Grenze der Gold-Schicht leicht prüfbar.


In [ ]:
DATA_DIR = project_path("DATA_DIR", "data")
CHECKPOINT_DIR = project_path("CHECKPOINT_DIR", "data/checkpoints")
GOLD_DIR = DATA_DIR / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"
CITY_METADATA_PATH = DATA_DIR / "silver" / "city_metadata.parquet"
EEA_DAILY_PATH = DATA_DIR / "silver" / "eea_city_daily.parquet"
OPEN_METEO_SILVER_PATH = DATA_DIR / "silver" / "open_meteo_city_hourly"
PHASE5_EVENTS_PATH = DATA_DIR / "bronze" / "open_meteo_raw" / "open_meteo_air_quality_events.jsonl"

DAILY_SUMMARY_PATH = GOLD_DIR / "city_air_quality_daily_summary.parquet"
RANKING_PATH = GOLD_DIR / "pollutant_ranking_by_city.parquet"
CONTEXT_PATH = GOLD_DIR / "city_context_air_quality.parquet"
LIVE_LATEST_PATH = GOLD_DIR / "live_air_quality_latest.parquet"
QUALITY_PATH = GOLD_DIR / "data_quality_summary.parquet"

print({"data_dir": str(DATA_DIR), "gold_dir": str(GOLD_DIR), "city_reference_exists": CITY_REFERENCE_PATH.exists(), "eea_daily_exists": EEA_DAILY_PATH.exists()})


### Infrastruktur- und Aussageparameter laden

Phase 7 liest Spark-Master, Kafka-Broker, Topic und optionale Speicherprobe, ohne einen Produzenten zu starten. `MIN_FINAL_HISTORY_DAYS` verhindert, dass ein kurzer technischer EEA-Smoke-Test fälschlich als belastbare historische Analyse freigegeben wird.


In [ ]:
SPARK_MASTER_URL = os.getenv("SPARK_MASTER_URL", "local[*]")
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "<kafka-host>:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_AIR_QUALITY_LIVE", "LIVE-bdeng_gXX_air_quality_live")
RUN_OPEN_METEO_KAFKA_PRODUCER = os.getenv("RUN_OPEN_METEO_KAFKA_PRODUCER", "false").lower() == "true"
RUN_PHASE7_SPARK_STORAGE_PROBE = os.getenv("RUN_PHASE7_SPARK_STORAGE_PROBE", "false").lower() == "true"
MIN_FINAL_HISTORY_DAYS = int(os.getenv("MIN_FINAL_HISTORY_DAYS", "365"))
assert MIN_FINAL_HISTORY_DAYS >= 1, "MIN_FINAL_HISTORY_DAYS muss mindestens 1 sein"

print({"spark_master_url": SPARK_MASTER_URL, "kafka_bootstrap_servers": KAFKA_BOOTSTRAP_SERVERS, "run_spark_storage_probe": RUN_PHASE7_SPARK_STORAGE_PROBE, "min_final_history_days": MIN_FINAL_HISTORY_DAYS})


### Leichtgewichtige TCP-Prüfung definieren

Die Prüfung weist Platzhalter und unerreichbare Endpunkte sichtbar als Daten aus.


In [ ]:
def tcp_check(endpoint: str, timeout_seconds: float = 2.0) -> dict:
    if "<" in endpoint or ":" not in endpoint:
        return {"endpoint": endpoint, "reachable": False, "reason": "Platzhalter oder fehlerhafter Endpunkt"}
    host, port = endpoint.rsplit(":", 1)
    try:
        with socket.create_connection((host, int(port)), timeout=timeout_seconds):
            return {"endpoint": endpoint, "reachable": True, "reason": None}
    except Exception as exc:
        return {"endpoint": endpoint, "reachable": False, "reason": str(exc)}

print("OK: tcp_check definiert")


### Kafka- und Spark-Endpunkte prüfen

Der lokale Spark-Modus gilt lokal als verfügbar. Entfernte Endpunkte erhalten ein explizites TCP-Prüfergebnis.


In [ ]:
spark_endpoint = SPARK_MASTER_URL.removeprefix("spark://") if SPARK_MASTER_URL.startswith("spark://") else SPARK_MASTER_URL
infrastructure_status = {
    "kafka": tcp_check(KAFKA_BOOTSTRAP_SERVERS),
    "spark_master": tcp_check(spark_endpoint) if SPARK_MASTER_URL.startswith("spark://") else {
        "endpoint": SPARK_MASTER_URL, "reachable": True, "reason": "Lokaler Spark-Modus"
    },
}
print({
    "project_root": str(PROJECT_ROOT),
    "spark_master_url": SPARK_MASTER_URL,
    "kafka_bootstrap_servers": KAFKA_BOOTSTRAP_SERVERS,
    "kafka_topic": KAFKA_TOPIC,
    "run_open_meteo_kafka_producer": RUN_OPEN_METEO_KAFKA_PRODUCER,
    "run_phase7_spark_storage_probe": RUN_PHASE7_SPARK_STORAGE_PROBE,
    "infrastructure_status": infrastructure_status,
})


## Umsetzung

### Bereitschaft der Phasen 0 bis 6 prüfen

Notebook `07` prüft Repository-Struktur, Abhängigkeiten, Schemas, Join-Schlüssel, Provenienz und optionalen Clusterspeicher. Fehlende Live-Silver-Daten aus Phase 6 wählen einen expliziten lokalen Rekonstruktionspfad.


#### Notebook-Reihenfolge und erforderliche Dateien prüfen

Zuerst wird geprüft, ob die Notebooks `00` bis `07`, verpflichtende Silver-Eingaben und kein versehentlich erzeugter Ordner `notebooks/data/` vorhanden sind.


In [ ]:
required_notebooks = [PROJECT_ROOT / "notebooks" / f"{index:02d}_{name}.ipynb" for index, name in [
    (0, "project_scope_and_requirements"),
    (1, "source_spike_and_cluster_check"),
    (2, "city_reference_model"),
    (3, "eea_batch_ingestion"),
    (4, "wikipedia_web_scraping"),
    (5, "open_meteo_api_and_kafka_producer"),
    (6, "spark_structured_streaming_kafka_to_parquet"),
    (7, "gold_layer_and_data_quality"),
]]
for path in required_notebooks:
    assert path.exists(), f"Notebook fehlt: {path}"
assert not (PROJECT_ROOT / "notebooks" / "data").exists(), "Falscher Ausgabeordner vorhanden: notebooks/data"
for path in [CITY_REFERENCE_PATH, CITY_METADATA_PATH, EEA_DAILY_PATH]:
    assert path.exists(), f"Erforderliche Silver-Eingabe fehlt: {path}"

print(f"OK: alle {len(required_notebooks)} Notebooks und 3 Silver-Eingaben vorhanden, kein notebooks/data-Fehler")


#### Silver-Datensätze laden

Die Gold-Schicht liest das gemeinsame Städtemodell, gescrapte Kontextdaten und historische tägliche EEA-Aggregate.


In [ ]:
city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
city_metadata_df = pd.read_parquet(CITY_METADATA_PATH)
eea_daily_df = pd.read_parquet(EEA_DAILY_PATH)

print(f"OK: Geladen — city_reference: {len(city_reference_df)} Zeilen, city_metadata: {len(city_metadata_df)} Zeilen, eea_daily: {len(eea_daily_df)} Zeilen")


#### Silver-Verträge und Joins validieren

Erforderliche Spalten, Eindeutigkeit der Städte, Schadstoffumfang und Verknüpfbarkeit werden vor dem Schreiben geprüft.


In [ ]:
required_contracts = {
    "city_reference": (city_reference_df, {"city_id", "city_name", "country_code", "latitude", "longitude"}),
    "city_metadata": (city_metadata_df, {"city_id", "population", "area_km2", "population_density", "parse_status"}),
    "eea_daily": (eea_daily_df, {"city_id", "date", "pollutant", "mean_value", "min_value", "max_value", "observation_count", "source", "data_status"}),
}
for name, (frame, columns) in required_contracts.items():
    missing = columns - set(frame.columns)
    assert not missing, f"{name} erforderliche Spalten fehlen: {sorted(missing)}"

assert len(city_reference_df) >= 8 and city_reference_df["city_id"].is_unique
assert city_metadata_df["city_id"].is_unique
assert set(eea_daily_df["city_id"]).issubset(set(city_reference_df["city_id"]))
assert set(city_metadata_df["city_id"]).issubset(set(city_reference_df["city_id"]))
assert set(eea_daily_df["pollutant"]).issubset({"pm2_5", "pm10", "no2"})

print(f"OK: alle Silver-Verträge bestanden — {len(required_contracts)} Datensätze, Schadstoffe: {sorted(eea_daily_df['pollutant'].unique())}")


#### Optional Spark-Worker-Speicher testen

Die Probe ist standardmäßig deaktiviert. In JupyterHub schreibt und liest sie mit dem konfigurierten Spark-Master einen kleinen Parquet-Datensatz.


In [ ]:
spark_storage_status = {"probe_requested": RUN_PHASE7_SPARK_STORAGE_PROBE, "passed": False, "reason": "nicht angefordert"}
if RUN_PHASE7_SPARK_STORAGE_PROBE:
    probe_path = DATA_DIR / "_phase7_spark_storage_probe"
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName("phase7-storage-probe").master(SPARK_MASTER_URL).getOrCreate()
        spark.createDataFrame([(1, "ok")], ["id", "status"]).write.mode("overwrite").parquet(str(probe_path))
        assert spark.read.parquet(str(probe_path)).count() == 1
        spark_storage_status = {"probe_requested": True, "passed": True, "reason": None}
    except Exception as exc:
        spark_storage_status = {"probe_requested": True, "passed": False, "reason": str(exc)}
    finally:
        try:
            spark.stop()
        except Exception:
            pass
        shutil.rmtree(probe_path, ignore_errors=True)

print({"spark_storage_status": spark_storage_status})


### Historischen EEA-Status und Analysezeitraum prüfen

Die EEA-Markierung `data_status` zeigt, ob reale API-Daten verwendet wurden. Zusätzlich werden Anzahl und Spanne der historischen Tage berechnet. Ein kurzer Smoke-Test bleibt ausführbar, erlaubt aber keine finalen empirischen Aussagen.


In [ ]:
real_eea_api_statuses = {"real_eea_api_postgres", "real_eea_api_parquet"}
eea_sample_fallback_used = not set(eea_daily_df["data_status"]).issubset(real_eea_api_statuses)
eea_dates = pd.to_datetime(eea_daily_df["date"])
historical_day_count = int(eea_dates.nunique())
historical_span_days = int((eea_dates.max() - eea_dates.min()).days + 1)
history_window_sufficient = historical_span_days >= MIN_FINAL_HISTORY_DAYS
historical_coverage_df = (
    eea_daily_df.groupby(["city_id", "pollutant"])["observation_count"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=["pm2_5", "pm10", "no2"], fill_value=0)
)
final_analytical_claims_allowed = not eea_sample_fallback_used and history_window_sufficient
print({
    "city_reference_rows": len(city_reference_df),
    "city_metadata_rows": len(city_metadata_df),
    "eea_daily_rows": len(eea_daily_df),
    "eea_data_status": sorted(eea_daily_df["data_status"].unique()),
    "eea_sample_fallback_used": eea_sample_fallback_used,
    "historical_day_count": historical_day_count,
    "historical_span_days": historical_span_days,
    "min_final_history_days": MIN_FINAL_HISTORY_DAYS,
    "history_window_sufficient": history_window_sufficient,
    "spark_storage_status": spark_storage_status,
})
display(historical_coverage_df)


### Historische Gold-Tabellen erstellen

Historische Gold-Tabellen verwenden ausschließlich tägliche EEA-Aggregate. Live-Werte von Open-Meteo werden nicht in historische Rankings gemischt.


#### Historische EEA-Werte mit Stadtkontext verbinden

Der historische Zweig nutzt nur tägliche EEA-Zeilen und ergänzt Anzeigefelder sowie Wikipedia-Kontext.


In [ ]:
eea_enriched_df = (
    eea_daily_df
    .merge(city_reference_df[["city_id", "city_name", "country_code", "latitude", "longitude"]], on="city_id", how="left", validate="many_to_one")
    .merge(city_metadata_df[["city_id", "population", "area_km2", "population_density", "parse_status"]], on="city_id", how="left", validate="many_to_one")
)
assert eea_enriched_df["city_name"].notna().all(), "EEA-Zeilen ohne Verknüpfung zur Städtereferenz gefunden"
eea_enriched_df["dataset_context"] = "eea_historical"

print(f"OK: eea_enriched_df — {len(eea_enriched_df)} Zeilen, city_reference und city_metadata verknüpft, dataset_context=eea_historical")


#### Historische tägliche Gold-Tabelle erstellen

Die Tabelle enthält je Stadt, Datum und Schadstoff genau eine prüfbare Zeile und benennt Kennzahlen für die Visualisierung um.


In [ ]:
city_air_quality_daily_summary_df = eea_enriched_df[[
    "city_id", "city_name", "country_code", "date", "pollutant", "unit",
    "mean_value", "min_value", "max_value", "observation_count", "source",
    "data_status", "dataset_context",
]].rename(columns={"mean_value": "avg_value", "observation_count": "measurement_count"})

print(f"OK: city_air_quality_daily_summary_df — {len(city_air_quality_daily_summary_df)} Zeilen, {len(city_air_quality_daily_summary_df.columns)} Spalten")


#### Schadstoffbezogene Städterankings berechnen

Mittelwerte, Bereiche, Beobachtungsanzahlen und ein dichtes Ranking bereiten den zentralen Vergleich für Phase 8 vor.


In [ ]:
pollutant_ranking_by_city_df = (
    city_air_quality_daily_summary_df
    .groupby(["city_id", "city_name", "country_code", "pollutant", "unit", "data_status", "dataset_context"], as_index=False)
    .agg(
        mean_pollutant_value=("avg_value", "mean"),
        min_pollutant_value=("min_value", "min"),
        max_pollutant_value=("max_value", "max"),
        daily_observation_count=("date", "count"),
        measurement_count=("measurement_count", "sum"),
    )
)
pollutant_ranking_by_city_df["pollutant_rank"] = (
    pollutant_ranking_by_city_df.groupby("pollutant")["mean_pollutant_value"]
    .rank(method="dense", ascending=False).astype(int)
)

print(f"OK: pollutant_ranking_by_city_df — {len(pollutant_ranking_by_city_df)} Zeilen, Rang 1 = höchste Belastung")


#### Wikipedia-Kontext zu Rankings ergänzen

Bevölkerung, Fläche und Dichte ermöglichen explorative Kontextdiagramme, ohne Kausalität zu behaupten.


In [ ]:
city_context_air_quality_df = pollutant_ranking_by_city_df.merge(
    city_metadata_df[["city_id", "population", "area_km2", "population_density", "parse_status"]],
    on="city_id", how="left", validate="many_to_one",
)

print(f"OK: city_context_air_quality_df — {len(city_context_air_quality_df)} Zeilen, Wikipedia-Kontext verknüpft")


#### Historische Gold-Parquets schreiben und erneut lesen

Jede historische Ausgabe wird gespeichert und unmittelbar erneut eingelesen.


In [ ]:
for frame, path in [
    (city_air_quality_daily_summary_df, DAILY_SUMMARY_PATH),
    (pollutant_ranking_by_city_df, RANKING_PATH),
    (city_context_air_quality_df, CONTEXT_PATH),
]:
    frame.to_parquet(path, index=False)
    assert len(pd.read_parquet(path)) == len(frame), f"Abweichung beim erneuten Einlesen aus Parquet: {path}"

print({
    "daily_summary_rows": len(city_air_quality_daily_summary_df),
    "ranking_rows": len(pollutant_ranking_by_city_df),
    "context_rows": len(city_context_air_quality_df),
})
pollutant_ranking_by_city_df.sort_values(["pollutant", "pollutant_rank"]).head(12)


### Getrennten Live-Snapshot fertigstellen

Silver-Parquet aus Phase 6 wird bevorzugt. Für lokale Reproduzierbarkeit können JSONL-Ereignisse aus Phase 5 den neuesten Snapshot rekonstruieren. `live_input_mode` dokumentiert den verwendeten Pfad.


#### Parquet-Leser definieren

Der kleine Wrapper hält den bevorzugten Silver-Pfad aus Phase 6 explizit sichtbar.


In [ ]:
def read_parquet_dataset(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)

print("OK: read_parquet_dataset definiert")


#### Phase-6-Silver oder lokalen JSONL-Fallback auswählen

Phase-6-Parquet wird bevorzugt. Der Rekonstruktionspfad aus Phase 5 ist nur als ausdrücklich markierter lokaler Fallback erlaubt.


In [ ]:
if OPEN_METEO_SILVER_PATH.exists():
    open_meteo_hourly_df = read_parquet_dataset(OPEN_METEO_SILVER_PATH)
    is_pandas_mock = set(open_meteo_hourly_df.get("processing_mode", [])) == {"pandas_mock_no_pyspark"}
    live_input_mode = "phase6_pandas_mock_no_pyspark_silver" if is_pandas_mock else "phase6_spark_stream_silver"
else:
    assert PHASE5_EVENTS_PATH.exists(), "Live-Fallback-JSONL fehlt. Führe Notebook 05 vor Notebook 07 aus."
    events = [json.loads(line) for line in PHASE5_EVENTS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
    open_meteo_hourly_df = pd.DataFrame(events)
    live_input_mode = "phase5_jsonl_mock_reconstruction"

print(f"OK: Live-Eingabe geladen — {len(open_meteo_hourly_df)} Ereignisse, live_input_mode={live_input_mode!r}")


#### Live-Ereignisse validieren und deduplizieren

Der Ereignisvertrag wird geprüft und Zeitstempel werden normalisiert, bevor je Stadt das neueste Ereignis ausgewählt wird.


In [ ]:
required_live_columns = {"event_id", "city_id", "event_time_utc", "ingestion_time_utc", "data_status", "pm2_5", "pm10", "no2"}
missing_live = required_live_columns - set(open_meteo_hourly_df.columns)
assert not missing_live, f"Live input erforderliche Spalten fehlen: {sorted(missing_live)}"
open_meteo_hourly_df["event_time_ts"] = pd.to_datetime(open_meteo_hourly_df["event_time_utc"], utc=True, errors="coerce")
open_meteo_hourly_df["ingestion_time_ts"] = pd.to_datetime(open_meteo_hourly_df["ingestion_time_utc"], utc=True, errors="coerce")
assert open_meteo_hourly_df[["event_time_ts", "ingestion_time_ts"]].notna().all().all()
open_meteo_hourly_df = open_meteo_hourly_df.drop_duplicates("event_id")

print(f"OK: Live-Ereignisse validiert — {len(open_meteo_hourly_df)} eindeutige Ereignisse, alle Zeitstempel gültig")


#### Getrennten Live-Snapshot schreiben

Der Snapshot wird mit Stadtkontext verknüpft, als `open_meteo_live` markiert, gespeichert und unabhängig von historischen Tabellen erneut eingelesen.


In [ ]:
live_event_columns = [
    "event_id", "city_id", "event_time_utc", "ingestion_time_utc", "event_time_ts",
    "ingestion_time_ts", "data_status", "pm2_5", "pm10", "no2",
]
live_air_quality_latest_df = (
    open_meteo_hourly_df[live_event_columns]
    .sort_values(["city_id", "event_time_ts", "ingestion_time_ts"], ascending=[True, False, False])
    .drop_duplicates("city_id", keep="first")
    .merge(city_reference_df[["city_id", "city_name", "country_code", "latitude", "longitude"]], on="city_id", how="left", validate="one_to_one")
    .merge(city_metadata_df[["city_id", "population", "area_km2", "population_density", "parse_status"]], on="city_id", how="left", validate="one_to_one")
)
live_air_quality_latest_df["dataset_context"] = "open_meteo_live"
live_air_quality_latest_df["live_input_mode"] = live_input_mode
if LIVE_LATEST_PATH.is_dir():
    shutil.rmtree(LIVE_LATEST_PATH)
elif LIVE_LATEST_PATH.exists():
    LIVE_LATEST_PATH.unlink()
live_air_quality_latest_df.to_parquet(LIVE_LATEST_PATH, index=False)
live_readback_df = pd.read_parquet(LIVE_LATEST_PATH)
assert live_readback_df["city_id"].is_unique
assert set(live_readback_df["dataset_context"]) == {"open_meteo_live"}
print({"live_input_mode": live_input_mode, "live_latest_rows": len(live_readback_df)})
live_readback_df[["city_id", "city_name", "event_time_ts", "data_status", "live_input_mode", "pm2_5", "pm10", "no2"]]


## Validierung und Qualitätsprüfungen

Der Qualitätsbericht profiliert jeden Gold-Datensatz und dokumentiert Integritätsprüfungen. Historische und Live-Provenienz bleiben sichtbar getrennt.


#### Profilierung auf Datensatzebene definieren

Das Profil fasst Zeilen, Spalten, doppelte Schlüssel, fehlende Werte, Stadtabdeckung und Provenienz je Gold-Datensatz zusammen.


In [ ]:
def dataset_profile(name: str, frame: pd.DataFrame, key_columns: list) -> dict:
    duplicate_keys = int(frame.duplicated(key_columns).sum())
    return {
        "dataset": name,
        "row_count": len(frame),
        "column_count": len(frame.columns),
        "duplicate_key_count": duplicate_keys,
        "missing_value_count": int(frame.isna().sum().sum()),
        "city_count": int(frame["city_id"].nunique()) if "city_id" in frame else None,
        "dataset_contexts": ",".join(sorted(frame["dataset_context"].dropna().astype(str).unique())) if "dataset_context" in frame else None,
        "data_statuses": ",".join(sorted(frame["data_status"].dropna().astype(str).unique())) if "data_status" in frame else None,
        "profiled_at_utc": datetime.now(timezone.utc).isoformat(),
    }

print("OK: dataset_profile definiert")


#### Alle Gold-Datensätze profilieren

Historische Tabellen und der getrennte Live-Snapshot werden mit passenden Schlüsseln bewertet.


In [ ]:
profiles = [
    dataset_profile("city_air_quality_daily_summary", city_air_quality_daily_summary_df, ["city_id", "date", "pollutant"]),
    dataset_profile("pollutant_ranking_by_city", pollutant_ranking_by_city_df, ["city_id", "pollutant"]),
    dataset_profile("city_context_air_quality", city_context_air_quality_df, ["city_id", "pollutant"]),
    dataset_profile("live_air_quality_latest", live_readback_df, ["city_id"]),
]
data_quality_summary_df = pd.DataFrame(profiles)

print(f"OK: {len(profiles)} Datensätze profiliert — Duplikate: {data_quality_summary_df['duplicate_key_count'].sum()}, fehlende Werte gesamt: {data_quality_summary_df['missing_value_count'].sum()}")


#### Plausibilität der Schadstoffwerte berechnen

Historische und Live-Werte werden unabhängig gegen dokumentierte Schadstoffbereiche geprüft.


In [ ]:
plausibility_limits = {"pm2_5": 1000, "pm10": 2000, "no2": 1000}
historical_invalid = sum(
    int(((city_air_quality_daily_summary_df["pollutant"] == pollutant) &
         ~city_air_quality_daily_summary_df["avg_value"].between(0, maximum)).sum())
    for pollutant, maximum in plausibility_limits.items()
)
live_invalid = sum(
    int((~live_readback_df[pollutant].between(0, maximum) & live_readback_df[pollutant].notna()).sum())
    for pollutant, maximum in plausibility_limits.items()
)

print(f"OK: Plausibilitätsprüfung — historisch unplausibel: {historical_invalid}, live unplausibel: {live_invalid}")


#### Invarianten der Gold-Schicht erzwingen

Assertions verhindern Duplikate, vermischten Kontext, unplausible Werte und Schreibvorgänge unter `notebooks/data/`.


In [ ]:
assert set(city_air_quality_daily_summary_df["dataset_context"]) == {"eea_historical"}
assert set(pollutant_ranking_by_city_df["dataset_context"]) == {"eea_historical"}
assert set(city_context_air_quality_df["dataset_context"]) == {"eea_historical"}
assert historical_invalid == 0, f"Ungültige historische Schadstoffzeilen: {historical_invalid}"
assert live_invalid == 0, f"Ungültige Live-Schadstoffzeilen: {live_invalid}"
assert not data_quality_summary_df["duplicate_key_count"].any(), data_quality_summary_df
assert not (PROJECT_ROOT / "notebooks" / "data").exists()

print("OK: alle Gold-Invarianten bestanden — dataset_context getrennt, keine Duplikate, alle Werte plausibel")


### Qualitätsbericht um Laufkontext ergänzen

Neben Zeilenzahlen und fehlenden Werten speichert der Bericht auch Infrastrukturstatus, Live-Herkunft und die historische Aussagegrenze. Dadurch bleibt maschinenlesbar sichtbar, ob ein Lauf nur die Mechanik demonstriert oder finale historische Aussagen tragen kann.


In [ ]:
data_quality_summary_df["eea_sample_fallback_used"] = eea_sample_fallback_used
data_quality_summary_df["historical_day_count"] = historical_day_count
data_quality_summary_df["historical_span_days"] = historical_span_days
data_quality_summary_df["min_final_history_days"] = MIN_FINAL_HISTORY_DAYS
data_quality_summary_df["history_window_sufficient"] = history_window_sufficient
data_quality_summary_df["final_analytical_claims_allowed"] = final_analytical_claims_allowed
data_quality_summary_df["live_input_mode"] = live_input_mode
data_quality_summary_df["kafka_tcp_reachable"] = infrastructure_status["kafka"]["reachable"]
data_quality_summary_df["spark_master_tcp_reachable"] = infrastructure_status["spark_master"]["reachable"]
data_quality_summary_df["spark_storage_probe_passed"] = spark_storage_status["passed"]
data_quality_summary_df.to_parquet(QUALITY_PATH, index=False)
quality_readback_df = pd.read_parquet(QUALITY_PATH)
quality_readback_df


### Gold-Ausgaben erneut lesen und an Phase 8 übergeben


#### Alle Gold-Ausgaben erneut lesen

Die Übergabe bestätigt, dass jede erwartete Gold-Datei existiert, lesbar ist und Zeilen enthält.


In [ ]:
gold_paths = {
    "city_air_quality_daily_summary": DAILY_SUMMARY_PATH,
    "pollutant_ranking_by_city": RANKING_PATH,
    "city_context_air_quality": CONTEXT_PATH,
    "live_air_quality_latest": LIVE_LATEST_PATH,
    "data_quality_summary": QUALITY_PATH,
}
gold_readback = {}
for name, path in gold_paths.items():
    assert path.exists(), f"Gold-Ausgabe fehlt: {path}"
    frame = pd.read_parquet(path)
    assert len(frame) > 0, f"Gold-Ausgabe ist leer: {path}"
    gold_readback[name] = {"rows": len(frame), "columns": list(frame.columns)}
print(gold_readback)


### übergabe an Phase 8 bewerten

Die übergabe prüft technische Verfügbarkeit und methodische Grenzen getrennt. Visualisierungen dürfen auch für einen Smoke-Test erzeugt werden. Finale empirische Aussagen sind erst bei realem EEA-Import und ausreichender historischer Spanne zulässig.


In [ ]:
phase8_readiness = {
    "historical_rankings_available": {"pollutant", "pollutant_rank", "mean_pollutant_value"}.issubset(pollutant_ranking_by_city_df.columns),
    "context_plot_available": {"population_density", "mean_pollutant_value"}.issubset(city_context_air_quality_df.columns),
    "live_snapshot_available": len(live_readback_df) > 0,
    "historical_and_live_separated": (
        set(city_context_air_quality_df["dataset_context"]) == {"eea_historical"}
        and set(live_readback_df["dataset_context"]) == {"open_meteo_live"}
    ),
    "history_window_sufficient": history_window_sufficient,
    "final_analytical_claims_allowed": final_analytical_claims_allowed,
}
assert all(value for key, value in phase8_readiness.items() if key not in {"history_window_sufficient", "final_analytical_claims_allowed"})
print({"phase8_readiness": phase8_readiness})
print("Phase 7 abgeschlossen. Phase 8 darf ausschließlich Gold-Ausgaben visualisieren.")


## Ergebnisse

Phase 7 erzeugt fünf Gold-Datensätze und einen sichtbaren Qualitätsbericht. Die Ausgabe dokumentiert explizit, ob reale EEA-Daten verfügbar sind, ob der historische Zeitraum ausreichend lang ist und ob der Live-Snapshot aus Spark-Silver-Parquet oder dem lokalen JSONL-Fallback stammt.

Bei `final_analytical_claims_allowed=False` darf Phase 8 die Visualisierungsmechanik zeigen, aber keine abschließenden empirischen Aussagen behaupten.


## Einschränkungen

- Ein kurzer EEA-API-Zeitraum demonstriert die Verarbeitung, erlaubt aber keine abschließenden historischen Aussagen.
- Fehlende Schadstoffabdeckung einzelner Städte bleibt sichtbar und wird nicht künstlich ergänzt.
- Wikipedia-Metadaten sind heuristischer Kontext; Korrelationen bleiben explorativ und nicht kausal.
- Open-Meteo-Live-Ereignisse sind Momentaufnahmen und bleiben von historischen EEA-Rankings getrennt.
- Eine JSONL-Rekonstruktion ist ein lokaler Fallback, kein Spark-aus-Kafka-Nachweis.
- Gemeinsamer FH-Clusterspeicher darf nur nach erfolgreicher Speicherprobe behauptet werden.
